In [1]:
from pathlib import Path
import sys

import torch
import torch.nn as nn
import torch.nn.functional as F

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from inference.decoding import multi_axis_decode
from inference.decoding import three_axis_refinement
from inference.sds import sds_refine_slice, sds_refine_volume
from loss import build_grayscale_tpc_target
from models import DDPM

LATENT_SHAPE = (4, 2, 2, 2)
VOLUME_SHAPE = (8, 1, 8, 8)
SEED = 0


In [2]:
class SmokeVAE(nn.Module):
    def encode(self, x):
        mu = F.avg_pool2d(x, kernel_size=4, stride=4).repeat(1, 4, 1, 1)
        return mu, torch.zeros_like(mu)

    def decode(self, z):
        return F.interpolate(z[:, :1], scale_factor=4, mode="nearest").clamp(0, 1)


class SmokeUNet(nn.Module):
    def forward(self, z_t, t):
        return torch.zeros_like(z_t)


torch.manual_seed(SEED)
vae = SmokeVAE()
unet = SmokeUNet()
ddpm = DDPM(timesteps=3)


In [3]:
volume_z = torch.randn(*LATENT_SHAPE)
volume = multi_axis_decode(vae, volume_z)
refined = three_axis_refinement(volume, vae, refinement_steps=1)

volume.shape, refined.shape, float(volume.min()), float(volume.max())


(torch.Size([8, 1, 8, 8]), torch.Size([8, 1, 8, 8]), 0.0, 0.8487103581428528)

In [4]:
target, bin_mat, bin_counts = build_grayscale_tpc_target(refined[3])
one_step, losses = sds_refine_slice(
    volume=refined,
    vae=vae,
    unet=unet,
    ddpm=ddpm,
    axis=0,
    index=3,
    lr=0.1,
    t_min=1,
    t_max=3,
    grayscale_tpc_target=target,
    grayscale_tpc_bin_mat=bin_mat,
    grayscale_tpc_bin_counts=bin_counts,
    grayscale_tpc_weight=1.0,
)

one_step.shape, {k: float(v) for k, v in losses.items()}, bool(torch.allclose(one_step[0], refined[0])), bool(torch.allclose(one_step[3], refined[3]))


(torch.Size([8, 1, 8, 8]),
 {'loss': -0.005116117652505636,
  'sds': -0.005116117652505636,
  'vf': 0.0,
  'tpc': 0.0,
  'rd': 0.0,
  'sa': 0.0},
 True,
 True)

In [5]:
final, history = sds_refine_volume(
    volume=refined,
    vae=vae,
    unet=unet,
    ddpm=ddpm,
    steps=2,
    lr=0.1,
    t_min=1,
    t_max=3,
    refinement_steps=1,
    generator=torch.Generator().manual_seed(SEED),
)

final.shape, len(history), history


(torch.Size([8, 1, 8, 8]),
 2,
 [{'loss': -0.009404907003045082,
   'sds': -0.009404907003045082,
   'vf': 0.0,
   'tpc': 0.0,
   'rd': 0.0,
   'sa': 0.0},
  {'loss': 0.0007983720861375332,
   'sds': 0.0007983720861375332,
   'vf': 0.0,
   'tpc': 0.0,
   'rd': 0.0,
   'sa': 0.0}])